# Care Collocates in Publication Corpus

In [ ]:
# Requires venv: open Terminal in this folder and run:
# >> source venv/bin/activate

import spacy
import os
import glob
from collections import Counter
import re

# Carica il modello inglese (usa "it_core_news_sm" se il corpus è in italiano)
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])  # disabilitiamo ciò che non serve, per velocità

In [2]:
CORPUS_FILE = "care-corpus.txt"
STOPWORDS_FILE = "stopwords.txt"

with open(STOPWORDS_FILE, encoding="utf-8") as f:
    stopwords = set(line.strip().lower() for line in f if line.strip())

with open(CORPUS_FILE, encoding="utf-8") as f:
    corpus_text = f.read()

print(f"Lunghezza corpus: {len(corpus_text)} caratteri")
print(f"Stopword caricate: {len(stopwords)}")

Lunghezza corpus: 6127036 caratteri
Stopword caricate: 153


In [3]:
nlp.max_length = len(corpus_text) + 1000

doc = nlp(corpus_text)

lemmas = [token.lemma_.lower() for token in doc if token.is_alpha]

print(f"Totale token processati: {len(lemmas)}")

Totale token processati: 735683


In [4]:
TARGET = "care"
WINDOW = 5

collocate_counter = Counter()

for i, lemma in enumerate(lemmas):
    if lemma == TARGET:
        start = max(0, i - WINDOW)
        end = min(len(lemmas), i + WINDOW + 1)
        context = lemmas[start:i] + lemmas[i+1:end]
        for w in context:
            if w != TARGET and w not in stopwords:
                collocate_counter[w] += 1

print(f"Occorrenze di '{TARGET}': {lemmas.count(TARGET)}")
print(f"Collocati distinti: {len(collocate_counter)}")

Occorrenze di 'care': 4607
Collocati distinti: 5522


In [5]:
import pandas as pd

df_collocates = pd.DataFrame(collocate_counter.most_common(), columns=["collocate", "frequenza"])
df_collocates.head(30)

,collocate,frequenza
0,health,965
1,heritage,369
2,take,281
3,collection,273
4,practice,266
5,cultural,224
6,museum,197
7,social,196
8,inward,163
9,community,161


In [7]:
df_collocates.sort_values("frequenza", ascending=False).to_csv("collocates_care.csv", index=False)
print("Salvato in collocates_care.csv")

Salvato in collocates_care.csv
